---
title: Construyendo una CNN desde cero con PyTorch
subject: Aprendizaje Profundo
subtitle: 
short_title: CNN desde cero con PyTorch
authors:
  - name: Jorge Anais
    orcid: 0000-0001-9051-1338
    email: jrganais@gmail.com
license: MIT
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jorgeanais/libro_aprendizaje_profundo/blob/main/cap2/040_CNN_MNIST.ipynb)

**Objetivo**: En este cuadernillo construirás una Red Neuronal Convolucional (CNN) desde cero usando PyTorch para clasificar imágenes del conjunto de datos MNIST. El desafío es lograr la mayor precisión posible en el conjunto de prueba.

Basado en Géron (2025) *Hands-On Machine Learning with Scikit-Learn and PyTorch*.

**Tip**: Las redes neuronales pueden ser muy lentas sin un acelerador de hardware. Si estás en colab, ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución y selecciona un acelerador de hardware GPU.

## Preparativos

In [ ]:
!pip install torchmetrics

In [ ]:
import sys
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# Verificar versiones
print(f"Python : {sys.version}")
print(f"PyTorch: {torch.__version__}")

# Selección automática de dispositivo
device = (
    "cuda" if torch.cuda.is_available()
    else "mps"  if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Usando dispositivo: {device}")

## Carga y exploración del conjunto de datos MNIST

MNIST contiene 70 000 imágenes en escala de grises de dígitos escritos a mano (0–9), cada una de 28×28 píxeles. Usaremos `torchvision` para descargarlo y prepararlo automáticamente.

In [ ]:
import torchvision
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.MNIST(
    root="datasets", train=True, download=True, transform=toTensor)
test_data = torchvision.datasets.MNIST(
    root="datasets", train=False, download=True, transform=toTensor)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000])

print(f"Muestras de entrenamiento : {len(train_data)}")
print(f"Muestras de validación    : {len(valid_data)}")
print(f"Muestras de prueba        : {len(test_data)}")

### Visualización del dataset

Antes de entrenar cualquier modelo es fundamental explorar los datos. Visualicemos algunas imágenes del conjunto de entrenamiento para entender con qué estamos trabajando.

In [ ]:
# Mostrar una cuadrícula de 5×10 con un ejemplo de cada dígito y varios más
class_names = [str(i) for i in range(10)]

fig, axes = plt.subplots(5, 10, figsize=(14, 7))
fig.suptitle("Muestra del conjunto MNIST (entrenamiento)", fontsize=14, fontweight="bold")

shown = {i: 0 for i in range(10)}
idx = 0
for img, label in train_data:
    row = shown[label]
    if row < 5:
        axes[row, label].imshow(img.squeeze(), cmap="gray")
        axes[row, label].axis("off")
        if row == 0:
            axes[row, label].set_title(f"Dígito {label}", fontsize=9)
        shown[label] += 1
    if all(v == 5 for v in shown.values()):
        break

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de clases en el conjunto de entrenamiento
all_labels = [label for _, label in train_data]
counts = np.bincount(all_labels)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(range(10), counts, color=plt.cm.tab10.colors)
ax.set_xticks(range(10))
ax.set_xticklabels([f"Dígito {i}" for i in range(10)], rotation=30)
ax.set_ylabel("Número de muestras")
ax.set_title("Distribución de clases — conjunto de entrenamiento")
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            str(count), ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()

## Preparación de DataLoaders

In [ ]:
from torch.utils.data import DataLoader

torch.manual_seed(42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader  = DataLoader(test_data,  batch_size=32)

# Verificar forma de un batch
X_batch, y_batch = next(iter(train_loader))
print(f"Forma del batch de imágenes  : {X_batch.shape}")
print(f"Forma del batch de etiquetas : {y_batch.shape}")

## Definición de la CNN

A continuación se proporciona una arquitectura de referencia que combina:

- **Capas convolucionales** con filtros 3×3 y relleno `same` para mantener las dimensiones espaciales.
- **MaxPool2d** con ventana 2×2 para reducir el tamaño a la mitad.
- **Capas densas** al final con `Dropout` para regularización.
- **`partial`** de `functools` para crear un constructor reutilizable de `Conv2d`.

> 💡 **Nota:** La primera capa usa `kernel_size=7` para capturar características de mayor escala al principio. Las capas posteriores usan `kernel_size=3` (valor por defecto definido en `DefaultConv2d`).

![arquitectura](https://github.com/jorgeanais/mlt2202/blob/main/ea2/s5/arquitecturared.png?raw=true)

In [ ]:
from functools import partial

torch.manual_seed(42)
DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding="same")

model = nn.Sequential(
    DefaultConv2d(in_channels=1, out_channels=64, kernel_size=7), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(in_channels=64, out_channels=128), nn.ReLU(),
    DefaultConv2d(in_channels=128, out_channels=128), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(in_channels=128, out_channels=256), nn.ReLU(),
    DefaultConv2d(in_channels=256, out_channels=256), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(in_features=2304, out_features=128), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features=128, out_features=64), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features=64, out_features=10),
).to(device)

print(model)

In [ ]:
# Contar parámetros entrenables
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total de parámetros entrenables: {total_params:,}")

## Funciones de entrenamiento y evaluación

Usaremos `torchmetrics` para calcular la precisión de manera eficiente, y definiremos un bucle de entrenamiento que registra métricas por época.

In [ ]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()


def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Época {epoch + 1:>2}/{n_epochs}  "
              f"pérdida entrenamiento: {history['train_losses'][-1]:.4f}  "
              f"exactitud entrenamiento: {history['train_metrics'][-1]:.4f}  "
              f"exactitud validación: {history['valid_metrics'][-1]:.4f}")
    return history

## Entrenamiento del modelo

In [ ]:
n_epochs = 20
optimizer = torch.optim.AdamW(model.parameters())
xentropy  = nn.CrossEntropyLoss()
accuracy  = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

history = train(model, optimizer, xentropy, accuracy,
                train_loader, valid_loader, n_epochs)

## Visualización del historial de entrenamiento

In [ ]:
epochs = range(1, n_epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pérdida
axes[0].plot(epochs, history["train_losses"], marker="o", label="Entrenamiento")
axes[0].set_title("Pérdida de entrenamiento")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend()
axes[0].grid(True)

# Exactitud
axes[1].plot(epochs, history["train_metrics"], marker="o", label="Entrenamiento")
axes[1].plot(epochs, history["valid_metrics"], marker="s", label="Validación")
axes[1].set_title("Exactitud por época")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Exactitud")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Evaluación final en el conjunto de prueba

In [ ]:
test_accuracy = evaluate_tm(model, test_loader, accuracy)
print(f"Exactitud en el conjunto de prueba: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

## Visualización de predicciones

Veamos cómo se desempeña el modelo en ejemplos individuales del conjunto de prueba, destacando en rojo las predicciones incorrectas.

In [ ]:
model.eval()
X_sample, y_sample = next(iter(test_loader))
with torch.no_grad():
    logits = model(X_sample.to(device))
    preds  = logits.argmax(dim=1).cpu()

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
fig.suptitle("Predicciones del modelo — conjunto de prueba\n(rojo = incorrecto)",
             fontsize=13, fontweight="bold")

for i, ax in enumerate(axes.flat):
    ax.imshow(X_sample[i].squeeze(), cmap="gray")
    ax.axis("off")
    color = "green" if preds[i] == y_sample[i] else "red"
    ax.set_title(f"P:{preds[i].item()} R:{y_sample[i].item()}",
                 fontsize=8, color=color)

plt.tight_layout()
plt.show()

## Matriz de confusión

In [ ]:
from torchmetrics.classification import MulticlassConfusionMatrix

conf_matrix = MulticlassConfusionMatrix(num_classes=10).to(device)
model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        conf_matrix.update(model(X_batch), y_batch)

cm = conf_matrix.compute().cpu().numpy()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(range(10))
ax.set_yticklabels(range(10))
ax.set_xlabel("Predicción", fontsize=12)
ax.set_ylabel("Etiqueta real", fontsize=12)
ax.set_title("Matriz de confusión — conjunto de prueba", fontsize=13)

for i in range(10):
    for j in range(10):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                fontsize=9, color=color)

plt.tight_layout()
plt.show()

---

###  Ejercicio 1. Preguntas de reflexión sobre la arquitectura 

 1. ¿Por qué se usa `padding="same"` en las capas convolucionales de `DefaultConv2d`? ¿Qué ventaja tiene frente a no usar relleno (`padding=0`)? 

<details>
<summary>👉 Clic para mostrar/ocultar respuesta</summary>
<p>
Al usar <code>padding="same"</code>, la capa convolucional agrega automáticamente el relleno necesario para que el mapa de características de salida tenga exactamente las mismas dimensiones espaciales que la entrada. Sin relleno, cada capa reduce la resolución (por ejemplo, con <code>kernel_size=3</code> y <code>padding=0</code> se pierde 1 píxel por borde). Esto facilita el diseño de arquitecturas más profundas sin perder información en los bordes, y hace que los cálculos de dimensiones sean más predecibles al apilar múltiples capas.
</p>
</details>



2. El modelo usa `Dropout(0.5)` en las capas densas. ¿Para qué sirve esta técnica y qué efecto tiene durante el entrenamiento vs. la inferencia? 

<details>
<summary>👉 Clic para mostrar/ocultar respuesta</summary>
<p>
El Dropout es una técnica de regularización que, durante el entrenamiento, apaga aleatoriamente una fracción (aquí el 50%) de las neuronas en cada paso. Esto obliga a la red a aprender representaciones más robustas y distribuidas, reduciendo la co-adaptación entre neuronas y así el sobreajuste. Durante la inferencia (modo <code>model.eval()</code>), el Dropout se desactiva automáticamente y todas las neuronas están activas; PyTorch escala los pesos para compensar la diferencia. Por eso es crucial llamar a <code>model.train()</code> durante el entrenamiento y <code>model.eval()</code> durante la evaluación.
</p>
</details>



### Ejercicio 2. Análisis de errores 

 Observa la matriz de confusión generada anteriormente y responde: 

1. ¿Qué par de dígitos confunde más el modelo? ¿Por qué crees que ocurre?
2. ¿Hay algún dígito que el modelo clasifique con muy alta precisión? ¿A qué podría deberse?

<details>
<summary>👉 Clic para mostrar/ocultar respuesta orientadora</summary>
<p>
Los pares más comúnmente confundidos en MNIST suelen ser 4↔9, 3↔5 y 7↔1, ya que comparten trazos estructuralmente similares. El dígito 0 suele clasificarse muy bien por ser visualmente distinto a todos los demás. Un análisis detallado de los errores puede guiar decisiones de diseño, como aplicar aumento de datos específico para las clases más confundidas.
</p>
</details>

In [ ]:
# Espacio para análisis adicional
# Por ejemplo: extraer y visualizar las imágenes que el modelo clasificó mal

all_preds, all_labels_list = [], []
all_imgs = []

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch.to(device))
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels_list.extend(y_batch.tolist())
        all_imgs.extend(X_batch)

# Imágenes mal clasificadas
errors = [(img, pred, real)
          for img, pred, real in zip(all_imgs, all_preds, all_labels_list)
          if pred != real]

print(f"Total de errores en test: {len(errors)} / {len(all_labels_list)}")

fig, axes = plt.subplots(3, 10, figsize=(14, 5))
fig.suptitle("Ejemplos mal clasificados", fontsize=12, fontweight="bold")
for i, ax in enumerate(axes.flat):
    if i >= len(errors):
        ax.axis("off")
        continue
    img, pred, real = errors[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.axis("off")
    ax.set_title(f"P:{pred}\nR:{real}", fontsize=7, color="red")
plt.tight_layout()
plt.show()

---

### Ejercicio 3. Mejora la arquitectura base

<font color="green">Experimenta modificando la arquitectura para intentar superar la exactitud del modelo base. Algunas ideas:</font>

- Agrega capas de **Batch Normalization** (`nn.BatchNorm2d`) después de cada convolución.
- Cambia el número de filtros o agrega más capas convolucionales.
- Prueba un optimizador diferente (por ejemplo, `torch.optim.SGD` con momentum).
- Añade **aumento de datos** (`T.RandomRotation`, `T.RandomAffine`) en el pipeline de transformaciones.
- Ajusta la tasa de aprendizaje con un `LRScheduler`.

Registra tus resultados en la siguiente celda de código:

In [ ]:
# =============================================
# TU CÓDIGO AQUÍ: Define tu propia arquitectura
# =============================================

torch.manual_seed(42)

# Ejemplo de punto de partida con BatchNorm:
# DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding="same")
# mi_modelo = nn.Sequential(
#     DefaultConv2d(in_channels=1, out_channels=32, kernel_size=3),
#     nn.BatchNorm2d(32), nn.ReLU(),
#     ...
# ).to(device)

# Entrena tu modelo:
# n_epochs_exp = 10
# optimizer_exp = torch.optim.AdamW(mi_modelo.parameters(), lr=1e-3)
# history_exp = train(mi_modelo, optimizer_exp, xentropy, accuracy,
#                     train_loader, valid_loader, n_epochs_exp)